In [2]:
import os
import json
from PIL import Image, ImageDraw
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF
from transformers import AutoProcessor, GitForCausalLM
from google.colab import drive


In [3]:
# Mount Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [4]:

# Paths
image_dir = "/content/drive/MyDrive/miniproject/Flicker8k_Dataset"
output_dir = "/content/drive/MyDrive/miniproject/json_outputs"
os.makedirs(output_dir, exist_ok=True)


In [5]:
# Device & models
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GitForCausalLM.from_pretrained("microsoft/git-large").to(device)
processor = AutoProcessor.from_pretrained("microsoft/git-large")
resnet = models.resnet50(pretrained=True).eval().to(device)
criterion = nn.CrossEntropyLoss()


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/2.82k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/453 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth

  0%|          | 0.00/97.8M [00:00<?, ?B/s]
  0%|          | 128k/97.8M [00:00<01:23, 1.22MB/s]
  0%|          | 384k/97.8M [00:00<01:05, 1.56MB/s]
  1%|          | 640k/97.8M [00:00<00:53, 1.90MB/s]
  3%|▎         | 3.12M/97.8M [00:00<00:09, 

In [6]:
# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


In [7]:
# Dummy ground truth loader
def get_ground_truth_captions(filename):
    return ["Ground truth caption 1", "Ground truth caption 2"]


In [8]:
# Caption generator
def generate_caption(img):
    img_tensor = processor(images=img, return_tensors="pt").to(device)
    outputs = model.generate(**img_tensor, max_length=50)
    return processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()


In [9]:


# FGSM
def fgsm_attack(img_tensor, model, label, epsilon=0.03):
    img_tensor.requires_grad = True
    output = model(img_tensor)
    loss = criterion(output, label)
    model.zero_grad()
    loss.backward()
    perturbed = img_tensor + epsilon * img_tensor.grad.sign()
    return torch.clamp(perturbed, 0, 1).detach()


In [10]:
# PGD
def pgd_attack(model, images, labels, eps=0.03, alpha=0.005, iters=10):
    ori = images.clone().detach()
    for _ in range(iters):
        images.requires_grad = True
        outputs = model(images)
        loss = criterion(outputs, labels)
        model.zero_grad()
        loss.backward()
        images = images + alpha * images.grad.sign()
        eta = torch.clamp(images - ori, min=-eps, max=eps)
        images = torch.clamp(ori + eta, 0, 1).detach()
    return images


In [11]:
# Adversarial patch
def apply_adversarial_patch(image, patch_size=60):
    img = image.copy()
    patch = Image.effect_noise((patch_size, patch_size), 100).convert("RGB")
    patch = TF.adjust_brightness(patch, 1.5)
    img.paste(patch, (img.width - patch_size, img.height - patch_size))
    return img


In [12]:
# Trigger patch
def add_trigger_patch(img, color=(255, 255, 0), size=40):
    img = img.copy()
    draw = ImageDraw.Draw(img)
    x0, y0 = img.width - size - 5, img.height - size - 5
    draw.rectangle([x0, y0, x0+size, y0+size], fill=color)
    return img


In [13]:

# Universal patch training
def train_universal_patch():
    from torchvision.datasets import CIFAR10
    cifar = CIFAR10(root="./data", train=True, download=True,
                    transform=transform)
    loader = DataLoader(cifar, batch_size=8, shuffle=True)
    patch = torch.rand(3, 50, 50, requires_grad=True, device=device)
    opt = torch.optim.Adam([patch], lr=0.1)
    for i, (imgs, labels) in enumerate(loader):
        if i > 100: break
        imgs, labels = imgs.to(device), labels.to(device)
        imgs[:, :, :50, :50] = patch.unsqueeze(0)
        output = resnet(imgs)
        loss = -criterion(output, labels)
        opt.zero_grad()
        loss.backward()
        opt.step()
    return patch.detach()


In [14]:

def apply_universal_patch(img, patch_tensor):
    img = img.resize((224, 224))
    img_tensor = transform(img).to(device)
    img_tensor[:, :50, :50] = patch_tensor
    return transforms.ToPILImage()(img_tensor.cpu())


In [15]:

# Train once
universal_patch = train_universal_patch()


100%|██████████| 170M/170M [00:03<00:00, 43.5MB/s]


In [16]:

# Batch config
batch_size = 100
all_images = [f for f in os.listdir(image_dir) if f.endswith(".jpg")]
total_batches = len(all_images) // batch_size + (len(all_images) % batch_size != 0)


In [17]:



# Process in batches
for batch_idx in range(total_batches):
    results = []
    batch_files = all_images[batch_idx*batch_size:(batch_idx+1)*batch_size]

    for filename in batch_files:
        try:
            path = os.path.join(image_dir, filename)
            image = Image.open(path).convert("RGB")
            ground_truth = get_ground_truth_captions(filename)

            # Original
            original_caption = generate_caption(image)

            # Label
            input_tensor = transform(image).unsqueeze(0).to(device).to(torch.float32)
            with torch.no_grad():
                label = resnet(input_tensor).argmax(dim=1)

            # FGSM
            fgsm_tensor = fgsm_attack(input_tensor.clone(), resnet, label)
            fgsm_image = transforms.ToPILImage()(fgsm_tensor.squeeze().cpu())
            fgsm_caption = generate_caption(fgsm_image)

            # PGD
            pgd_tensor = pgd_attack(resnet, input_tensor.clone(), label)
            pgd_image = transforms.ToPILImage()(pgd_tensor.squeeze().cpu())
            pgd_caption = generate_caption(pgd_image)

            # Adversarial Patch
            adv_patch_image = apply_adversarial_patch(image)
            a_caption = generate_caption(adv_patch_image)

            # Trigger Patch
            trigger_patch_image = add_trigger_patch(image)
            t_caption = generate_caption(trigger_patch_image)

            # Universal Patch
            universal_patch_image = apply_universal_patch(image, universal_patch)
            u_caption = generate_caption(universal_patch_image)

            # Save result
            results.append({
                "image_path": path,
                "ground_truth_captions": ground_truth,
                "original_caption": original_caption,
                "fgsm": fgsm_caption,
                "pgd": pgd_caption,
                "adversarial_patch": a_caption,
                "trigger_patch": t_caption,
                "universal_patch": u_caption
            })

            print(f"✅ Processed: {filename}")

        except Exception as e:
            print(f"❌ Error processing {filename}: {e}")

    # Save batch result
    batch_json_path = os.path.join(output_dir, f"captions_batch_{batch_idx+1}.json")
    with open(batch_json_path, "w") as f:
        json.dump(results, f, indent=2)
    print(f"\n📁 Saved batch {batch_idx+1} to {batch_json_path}")


✅ Processed: 378453580_21d688748e.jpg
✅ Processed: 377872672_d499aae449.jpg
✅ Processed: 380034515_4fbdfa6b26.jpg
✅ Processed: 380041023_0dfd712ef1.jpg
✅ Processed: 380527679_574749123d.jpg
✅ Processed: 381239475_044cbffa2b.jpg
✅ Processed: 381514859_b40418d9c3.jpg
✅ Processed: 381976882_0063d16d88.jpg
✅ Processed: 381275595_b429fd1639.jpg
✅ Processed: 382090166_be2c2c63e1.jpg
✅ Processed: 381052465_722e00807b.jpg
✅ Processed: 380590140_25b9889772.jpg
✅ Processed: 380515798_c2abbf46b0.jpg
✅ Processed: 380537190_11d6c0a412.jpg
✅ Processed: 384577800_fc325af410.jpg
✅ Processed: 383223174_7165a54c30.jpg
✅ Processed: 386470686_1ae9242878.jpg
✅ Processed: 384465575_31294122c0.jpg
✅ Processed: 384465370_9918873f9a.jpg
✅ Processed: 386160015_d4b31df68e.jpg
✅ Processed: 385835044_4aa11f6990.jpg
✅ Processed: 385186343_464f5fc186.jpg
✅ Processed: 382701159_f98c1988cd.jpg
✅ Processed: 382151094_c7376cf22b.jpg
✅ Processed: 387078972_514a38dc33.jpg
✅ Processed: 388386075_9ac3a89ada.jpg
✅ Processed:

KeyboardInterrupt: 

In [18]:
!pip install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 57.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling

In [1]:
!git clone https://github.com/salaniz/pycocoevalcap

Cloning into 'pycocoevalcap'...
remote: Enumerating objects: 821, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 821 (delta 4), reused 3 (delta 3), pack-reused 809 (from 2)
Receiving objects: 100% (821/821), 130.06 MiB | 13.92 MiB/s, done.
Resolving deltas: 100% (424/424), done.
Updating files: 100% (40/40), done.


In [2]:

!pip install pycocoevalcap/


Processing ./pycocoevalcap
  Preparing metadata (setup.py) ... done
  Created wheel for pycocoevalcap: filename=pycocoevalcap-1.2-py3-none-any.whl size=104312245 sha256=2de068ca89dfc4d56f8d3a246371c388f07bd7ca046efa57cd521c35c3c32c67
  Stored in directory: /tmp/pip-ephem-wheel-cache-27u7kgmu/wheels/0e/98/9f/b6578f2310a0adf702387edf950a2ba69dbf680c0b6830b312
Successfully built pycocoevalcap


In [3]:

import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.cider.cider import Cider
from bert_score import score as bert_score


In [4]:

# Load your JSON file
with open("abhi.json") as f:
    data = json.load(f)

smoother = SmoothingFunction().method4
meteor_scorer = Meteor()
cider_scorer = Cider()


In [5]:
# Initialize storage for scores
scores = {
    "original": {"BLEU": [], "METEOR": [], "CIDEr": []},
    "fgsm": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "pgd": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "adversarial_patch": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "trigger_patch": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []},
    "universal_patch": {"BLEU": [], "METEOR": [], "CIDEr": [], "BERT": []}
}

# List of attacks to evaluate
attacks = ["fgsm", "pgd", "adversarial_patch", "trigger_patch",  "universal_patch"]


In [6]:
def avg(lst): return round(np.mean(lst), 4)

# ORIGINAL CAPTION EVALUATION
print("\n🔍 Evaluating Original Captions...")
for idx, item in enumerate(data, 1):
    refs = [caption.lower().split() for caption in item["ground_truth_captions"]]
    orig = item["original_caption"].lower().split()

    scores["original"]["BLEU"].append(sentence_bleu(refs, orig, smoothing_function=smoother))
    scores["original"]["METEOR"].append(
        meteor_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [item["original_caption"]]})[0]
    )
    scores["original"]["CIDEr"].append(
        cider_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [item["original_caption"]]})[0]
    )

    if idx % 50 == 0 or idx == len(data):
        print(f"✅ Processed {idx}/{len(data)} original captions")

# Print Original Scores
print("\n📊 Original Caption Scores:")
print(f"  BLEU: {avg(scores['original']['BLEU'])}")
print(f"  METEOR: {avg(scores['original']['METEOR'])}")
print(f"  CIDEr: {avg(scores['original']['CIDEr'])}")



🔍 Evaluating Original Captions...
✅ Processed 50/1000 original captions
✅ Processed 100/1000 original captions
✅ Processed 150/1000 original captions
✅ Processed 200/1000 original captions
✅ Processed 250/1000 original captions
✅ Processed 300/1000 original captions
✅ Processed 350/1000 original captions
✅ Processed 400/1000 original captions
✅ Processed 450/1000 original captions
✅ Processed 500/1000 original captions
✅ Processed 550/1000 original captions
✅ Processed 600/1000 original captions
✅ Processed 650/1000 original captions
✅ Processed 700/1000 original captions
✅ Processed 750/1000 original captions
✅ Processed 800/1000 original captions
✅ Processed 850/1000 original captions
✅ Processed 900/1000 original captions
✅ Processed 950/1000 original captions
✅ Processed 1000/1000 original captions

📊 Original Caption Scores:
  BLEU: 0.1502
  METEOR: 0.1843
  CIDEr: 0.0


In [ ]:
for atk in attacks:
    print(f"\n⚔️ Evaluating {atk.upper()} Attack...")
    for idx, item in enumerate(data, 1):
        refs = [caption.lower().split() for caption in item["ground_truth_captions"]]
        adv_cap = item[f"{atk}"]
        adv_tok = adv_cap.lower().split()

        scores[atk]["BLEU"].append(sentence_bleu(refs, adv_tok, smoothing_function=smoother))
        scores[atk]["METEOR"].append(
            meteor_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [adv_cap]})[0]
        )
        scores[atk]["CIDEr"].append(
            cider_scorer.compute_score({0: item["ground_truth_captions"]}, {0: [adv_cap]})[0]
        )

        # BERTScore: compare with original
        P, R, F1 = bert_score([adv_cap], [item["original_caption"]], lang="en", verbose=False)
        scores[atk]["BERT"].append(F1.item())

        if idx % 50 == 0 or idx == len(data):
            print(f"✅ Processed {idx}/{len(data)} for {atk}")

    # Print Attack Scores
    print(f"\n📊 Scores under {atk.upper()} Attack:")
    print(f"  BLEU: {avg(scores[atk]['BLEU'])} (drop: {round(avg(scores['original']['BLEU']) - avg(scores[atk]['BLEU']), 4)})")
    print(f"  METEOR: {avg(scores[atk]['METEOR'])} (drop: {round(avg(scores['original']['METEOR']) - avg(scores[atk]['METEOR']), 4)})")
    print(f"  CIDEr: {avg(scores[atk]['CIDEr'])} (drop: {round(avg(scores['original']['CIDEr']) - avg(scores[atk]['CIDEr']), 4)})")
    print(f"  BERTScore F1 vs Original: {avg(scores[atk]['BERT'])}")



⚔️ Evaluating FGSM Attack...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

...

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 200/1000 for adversarial_patch


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

✅ Processed 250/1000 for adversarial_patch


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho